# 🔢 Reconocimiento de Dígitos Escritos a Mano
## Ejercicio de Inteligencia Artificial con Redes Neuronales

---

En este ejercicio vamos a construir un modelo de **Inteligencia Artificial** capaz de reconocer dígitos escritos a mano (del 0 al 9).

Usaremos el famoso dataset **MNIST**, que contiene 70.000 imágenes de dígitos escritos a mano.

### 🎯 Objetivos de aprendizaje:
- Entender qué es una red neuronal
- Preparar y visualizar datos de imágenes
- Entrenar un modelo de clasificación
- Evaluar su rendimiento
- Hacer predicciones con imágenes nuevas

### 🗺️ Estructura del ejercicio:
1. Importar librerías
2. Cargar y explorar los datos
3. Preprocesar los datos
4. Construir la red neuronal
5. Entrenar el modelo
6. Evaluar el modelo
7. Hacer predicciones
8. 🏆 Desafíos extra

---
## 📦 Paso 1: Importar librerías

Primero importamos todas las herramientas que vamos a necesitar.

In [ ]:
# Librerías para manejo de datos y visualización
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow y Keras para construir la red neuronal
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Para evaluar el modelo
from sklearn.metrics import confusion_matrix, classification_report

print('✅ Librerías importadas correctamente')
print(f'📌 TensorFlow versión: {tf.__version__}')

---
## 📂 Paso 2: Cargar y explorar los datos

El dataset **MNIST** ya viene incluido en Keras. Contiene:
- **60.000 imágenes** para entrenar el modelo
- **10.000 imágenes** para evaluar el modelo

Cada imagen es de **28×28 píxeles** en escala de grises.

In [ ]:
# Cargar el dataset MNIST
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print('📊 Información del dataset:')
print(f'   Imágenes de entrenamiento: {X_train.shape}')
print(f'   Etiquetas de entrenamiento: {y_train.shape}')
print(f'   Imágenes de prueba:        {X_test.shape}')
print(f'   Etiquetas de prueba:       {y_test.shape}')
print(f'\n   Cada imagen es de {X_train.shape[1]}x{X_train.shape[2]} píxeles')
print(f'   Valores de píxel: de {X_train.min()} a {X_train.max()}')
print(f'   Clases (dígitos): {np.unique(y_train)}')

In [ ]:
# Visualizar algunas imágenes del dataset
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
fig.suptitle('Ejemplos del dataset MNIST', fontsize=14, fontweight='bold')

for digito in range(10):
    # Mostrar 3 ejemplos de cada dígito
    indices = np.where(y_train == digito)[0][:3]
    for fila, idx in enumerate(indices):
        axes[fila, digito].imshow(X_train[idx], cmap='gray')
        axes[fila, digito].set_title(f'Dígito: {digito}', fontsize=8)
        axes[fila, digito].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de dígitos en el conjunto de entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Histograma
digitos, conteos = np.unique(y_train, return_counts=True)
ax1.bar(digitos, conteos, color='steelblue', edgecolor='white')
ax1.set_title('Distribución de dígitos (Entrenamiento)', fontweight='bold')
ax1.set_xlabel('Dígito')
ax1.set_ylabel('Cantidad de imágenes')
ax1.set_xticks(range(10))
for i, (d, c) in enumerate(zip(digitos, conteos)):
    ax1.text(d, c + 50, str(c), ha='center', fontsize=9)

# Ejemplo de un dígito con sus píxeles
ax2.imshow(X_train[0], cmap='gray')
ax2.set_title(f'Ejemplo ampliado - Dígito: {y_train[0]}\n(imagen de 28x28 píxeles)', fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()

print('\n💡 El dataset está bastante balanceado: hay entre 5.000 y 7.000 imágenes de cada dígito.')

---
## ⚙️ Paso 3: Preprocesar los datos

Antes de entrenar, debemos preparar los datos:

1. **Normalizar**: los píxeles van de 0 a 255. Los dividimos por 255 para que queden entre 0 y 1. Esto ayuda a la red neuronal a aprender más rápido.
2. **Aplanar**: cada imagen de 28×28 la convertimos en un vector de 784 valores (para la arquitectura básica).

In [ ]:
# ── Normalización ──────────────────────────────────────────────
X_train_norm = X_train / 255.0
X_test_norm  = X_test  / 255.0

print('Antes de normalizar:')
print(f'  Valor mínimo: {X_train.min()},  Valor máximo: {X_train.max()}')
print('\nDespués de normalizar:')
print(f'  Valor mínimo: {X_train_norm.min():.2f},  Valor máximo: {X_train_norm.max():.2f}')

# ── Verificar forma ────────────────────────────────────────────
print(f'\nForma de X_train_norm: {X_train_norm.shape}  → (muestras, alto, ancho)')
print('✅ Datos normalizados correctamente')

---
## 🧠 Paso 4: Construir la Red Neuronal

Vamos a construir una red neuronal **densa** (también llamada *Fully Connected* o *MLP - Multilayer Perceptron*).

### Arquitectura:
```
Entrada (28x28 píxeles)
    ↓  Flatten (aplanar a 784 valores)
    ↓  Capa Densa 1: 256 neuronas + ReLU
    ↓  Dropout 0.3 (regularización)
    ↓  Capa Densa 2: 128 neuronas + ReLU
    ↓  Dropout 0.2
    ↓  Capa de Salida: 10 neuronas + Softmax
        (una por cada dígito del 0 al 9)
```

### Conceptos clave:
- **ReLU**: función de activación que introduce no-linealidad
- **Softmax**: convierte la salida en probabilidades (suman 1)
- **Dropout**: apaga neuronas aleatoriamente para evitar sobreajuste

In [ ]:
# Fijar semilla para reproducibilidad
tf.random.set_seed(42)
np.random.seed(42)

# Construir el modelo
modelo = keras.Sequential([
    # Capa de entrada: aplana la imagen 28x28 a un vector de 784
    layers.Flatten(input_shape=(28, 28), name='entrada'),

    # Primera capa oculta
    layers.Dense(256, activation='relu', name='oculta_1'),
    layers.Dropout(0.3, name='dropout_1'),   # Evita el sobreajuste

    # Segunda capa oculta
    layers.Dense(128, activation='relu', name='oculta_2'),
    layers.Dropout(0.2, name='dropout_2'),

    # Capa de salida: 10 neuronas (una por dígito), con Softmax
    layers.Dense(10, activation='softmax', name='salida')
], name='Reconocedor_Digitos')

# Resumen de la arquitectura
modelo.summary()

In [ ]:
# Compilar el modelo
modelo.compile(
    optimizer='adam',                          # Algoritmo de optimización
    loss='sparse_categorical_crossentropy',    # Función de pérdida para clasificación
    metrics=['accuracy']                       # Métrica a monitorear
)

print('✅ Modelo compilado')
print('\n📖 Parámetros de compilación:')
print('   Optimizer : Adam  → ajusta los pesos automáticamente')
print('   Loss      : Sparse Categorical Crossentropy  → mide el error en clasificación')
print('   Métrica   : Accuracy  → porcentaje de aciertos')

---
## 🏋️ Paso 5: Entrenar el Modelo

El entrenamiento consiste en mostrarle las imágenes al modelo muchas veces (**épocas**) para que ajuste sus parámetros internos y aprenda a reconocer los dígitos.

Usamos **Early Stopping** para detener el entrenamiento automáticamente si el modelo deja de mejorar.

In [ ]:
# Callback de Early Stopping
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,           # Espera 5 épocas sin mejora antes de parar
    restore_best_weights=True,
    verbose=1
)

# Entrenar
print('🚀 Iniciando entrenamiento...\n')
historia = modelo.fit(
    X_train_norm, y_train,
    epochs=30,
    batch_size=128,
    validation_split=0.15,   # 15% de entrenamiento se usa para validación
    callbacks=[early_stop],
    verbose=1
)

print('\n✅ Entrenamiento finalizado')

In [ ]:
# Visualizar la evolución del entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Evolución del Entrenamiento', fontsize=14, fontweight='bold')

epocas = range(1, len(historia.history['accuracy']) + 1)

# Accuracy
ax1.plot(epocas, historia.history['accuracy'],     label='Entrenamiento', color='steelblue', linewidth=2)
ax1.plot(epocas, historia.history['val_accuracy'], label='Validación',    color='orange',    linewidth=2, linestyle='--')
ax1.set_title('Exactitud (Accuracy)')
ax1.set_xlabel('Época')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1])

# Loss
ax2.plot(epocas, historia.history['loss'],     label='Entrenamiento', color='steelblue', linewidth=2)
ax2.plot(epocas, historia.history['val_loss'], label='Validación',    color='orange',    linewidth=2, linestyle='--')
ax2.set_title('Pérdida (Loss)')
ax2.set_xlabel('Época')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('💡 Si las curvas de entrenamiento y validación son similares → el modelo generaliza bien')
print('   Si training >> validation → sobreajuste (overfitting)')

---
## 📊 Paso 6: Evaluar el Modelo

Evaluamos el modelo con las **imágenes de prueba** que nunca vio durante el entrenamiento.

In [ ]:
# Evaluación en el conjunto de prueba
loss_test, acc_test = modelo.evaluate(X_test_norm, y_test, verbose=0)

print('=' * 45)
print('       RESULTADOS EN CONJUNTO DE PRUEBA')
print('=' * 45)
print(f'   Exactitud (Accuracy): {acc_test*100:.2f}%')
print(f'   Pérdida (Loss):       {loss_test:.4f}')
print('=' * 45)
print(f'\n🎯 El modelo acierta en {acc_test*100:.1f}% de los dígitos que nunca había visto!')

In [ ]:
# Obtener predicciones
y_pred_probs = modelo.predict(X_test_norm, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=range(10), yticklabels=range(10),
    ax=ax
)
ax.set_title('Matriz de Confusión\n(filas = real, columnas = predicho)', fontsize=13, fontweight='bold')
ax.set_xlabel('Dígito Predicho', fontsize=12)
ax.set_ylabel('Dígito Real',     fontsize=12)
plt.tight_layout()
plt.show()

print('\n💡 La diagonal principal muestra los aciertos.')
print('   Los valores fuera de la diagonal son errores.')

In [ ]:
# Reporte de clasificación completo
print('Reporte de clasificación por dígito:\n')
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)]))

In [ ]:
# Visualizar ejemplos de aciertos y errores
aciertos = np.where(y_pred == y_test)[0]
errores  = np.where(y_pred != y_test)[0]

fig, axes = plt.subplots(2, 10, figsize=(18, 4))
fig.suptitle('Aciertos ✅ (fila 1) vs Errores ❌ (fila 2)', fontsize=13, fontweight='bold')

for i in range(10):
    # Aciertos
    idx = aciertos[i]
    axes[0, i].imshow(X_test[idx], cmap='gray')
    axes[0, i].set_title(f'✅ {y_pred[idx]}', color='green', fontsize=9)
    axes[0, i].axis('off')

    # Errores
    idx = errores[i]
    axes[1, i].imshow(X_test[idx], cmap='gray')
    axes[1, i].set_title(f'❌ Real:{y_test[idx]}\n   Pred:{y_pred[idx]}', color='red', fontsize=8)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

---
## 🔮 Paso 7: Hacer Predicciones

Vamos a ver cómo el modelo predice dígitos individuales y qué tan seguro está de su respuesta.

In [ ]:
def predecir_digito(indice):
    """Muestra una imagen y las probabilidades de predicción del modelo."""
    imagen = X_test_norm[indice]
    real   = y_test[indice]

    # Predicción
    probs = modelo.predict(imagen[np.newaxis, ...], verbose=0)[0]
    pred  = np.argmax(probs)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

    # Imagen
    ax1.imshow(imagen, cmap='gray')
    color = 'green' if pred == real else 'red'
    ax1.set_title(f'Real: {real}  |  Predicción: {pred}', color=color, fontweight='bold', fontsize=12)
    ax1.axis('off')

    # Probabilidades
    colores = ['steelblue'] * 10
    colores[pred] = 'green' if pred == real else 'red'
    bars = ax2.bar(range(10), probs * 100, color=colores, edgecolor='white')
    ax2.set_title('Probabilidades por dígito (%)', fontweight='bold')
    ax2.set_xlabel('Dígito')
    ax2.set_ylabel('Probabilidad (%)')
    ax2.set_xticks(range(10))
    ax2.set_ylim([0, 105])
    for bar, prob in zip(bars, probs):
        if prob > 0.01:
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{prob*100:.1f}%', ha='center', va='bottom', fontsize=8)
    ax2.grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()
    print(f'Confianza del modelo: {probs[pred]*100:.1f}%')


# ── Probar con varios índices ──────────────────────────────────
for idx in [0, 100, 500, 1000, 2000]:
    print(f'\n--- Imagen #{idx} ---')
    predecir_digito(idx)

In [ ]:
# 🔧 ¡Probá con un índice a tu elección!
# Cambiá el número entre 0 y 9999

INDICE = 42   # ← modificá este valor

predecir_digito(INDICE)

---
## 🏆 Paso 8: Desafíos Extra

¿Terminaste el ejercicio base? ¡Probá estos desafíos para seguir aprendiendo!

---

### 🥉 Desafío 1 (Fácil): Cambiar la arquitectura
Modificá el modelo: agregá o quitá capas, cambiá la cantidad de neuronas, o modificá el Dropout.  
¿Mejora o empeora el accuracy?

In [ ]:
# DESAFÍO 1: Modificá la arquitectura y entrenala de nuevo
# TODO: modificá la cantidad de neuronas, capas o dropout

modelo_v2 = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    # ---- TU ARQUITECTURA ACÁ ----
    layers.Dense(128, activation='relu'),
    layers.Dense(64,  activation='relu'),
    # -----------------------------
    layers.Dense(10, activation='softmax')
])

modelo_v2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

historia_v2 = modelo_v2.fit(
    X_train_norm, y_train,
    epochs=15, batch_size=128,
    validation_split=0.15, verbose=1
)

_, acc_v2 = modelo_v2.evaluate(X_test_norm, y_test, verbose=0)
print(f'\nModelo original  → Accuracy: {acc_test*100:.2f}%')
print(f'Modelo v2        → Accuracy: {acc_v2*100:.2f}%')
print(f'Diferencia       → {(acc_v2 - acc_test)*100:+.2f}%')

---
### 🥈 Desafío 2 (Medio): Red Neuronal Convolucional (CNN)

Las **CNN** son mucho más potentes para procesar imágenes porque explotan la estructura espacial.
Implementá una CNN y compará los resultados.

In [ ]:
# DESAFÍO 2: Red Neuronal Convolucional

# Las CNN necesitan las imágenes con un canal de color extra
X_train_cnn = X_train_norm[..., np.newaxis]  # (60000, 28, 28, 1)
X_test_cnn  = X_test_norm[..., np.newaxis]

modelo_cnn = keras.Sequential([
    # Bloque convolucional 1
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),

    # Bloque convolucional 2
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Clasificador
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
], name='CNN_Digitos')

modelo_cnn.summary()

modelo_cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

historia_cnn = modelo_cnn.fit(
    X_train_cnn, y_train,
    epochs=15, batch_size=128,
    validation_split=0.15,
    callbacks=[early_stop], verbose=1
)

_, acc_cnn = modelo_cnn.evaluate(X_test_cnn, y_test, verbose=0)
print(f'\nRed Densa (MLP) → Accuracy: {acc_test*100:.2f}%')
print(f'Red CNN         → Accuracy: {acc_cnn*100:.2f}%')
print(f'Mejora CNN      → {(acc_cnn - acc_test)*100:+.2f}%')

---
### 🥇 Desafío 3 (Difícil): Dibujar tu propio dígito

Dibujá un dígito en el canvas y hacé que el modelo lo reconozca en tiempo real.

In [ ]:
# DESAFÍO 3: Canvas interactivo para dibujar y predecir

from IPython.display import display, HTML, Javascript
import base64
from PIL import Image
import io

canvas_html = """
<div style="text-align:center; font-family: Arial;">
  <h3>✏️ Dibujá un dígito (0-9)</h3>
  <canvas id="canvas" width="280" height="280"
          style="border:3px solid #333; cursor:crosshair; background:black;"></canvas>
  <br><br>
  <button onclick="limpiar()" style="margin:5px; padding:8px 20px;">🗑️ Limpiar</button>
  <button onclick="guardar()" style="margin:5px; padding:8px 20px; background:#4CAF50; color:white;">🔮 Predecir</button>
  <p id="resultado" style="font-size:20px; font-weight:bold; color:#333;"></p>
</div>

<script>
var canvas  = document.getElementById('canvas');
var ctx     = canvas.getContext('2d');
var dibujando = false;

ctx.strokeStyle = 'white';
ctx.lineWidth   = 20;
ctx.lineCap     = 'round';

canvas.addEventListener('mousedown', e => { dibujando = true; ctx.beginPath(); ctx.moveTo(e.offsetX, e.offsetY); });
canvas.addEventListener('mousemove', e => { if (dibujando) { ctx.lineTo(e.offsetX, e.offsetY); ctx.stroke(); } });
canvas.addEventListener('mouseup',   () => { dibujando = false; });

function limpiar() { ctx.clearRect(0, 0, 280, 280); document.getElementById('resultado').innerText = ''; }

function guardar() {
    var data = canvas.toDataURL('image/png');
    // Guardamos en una variable de Python via kernel
    var kernel = IPython.notebook.kernel;
    kernel.execute("imagen_canvas = '" + data + "'");
    setTimeout(() => {
        kernel.execute("predecir_desde_canvas()");
    }, 500);
}
</script>
"""

display(HTML(canvas_html))


def predecir_desde_canvas():
    """Procesa la imagen del canvas y hace una predicción."""
    import base64
    from PIL import Image
    import io

    global imagen_canvas

    # Decodificar base64
    header, data = imagen_canvas.split(',', 1)
    img_bytes = base64.b64decode(data)
    img = Image.open(io.BytesIO(img_bytes)).convert('L')  # Escala de grises
    img = img.resize((28, 28), Image.LANCZOS)

    # Convertir a array y normalizar
    arr = np.array(img) / 255.0

    # Mostrar la imagen procesada
    plt.figure(figsize=(2, 2))
    plt.imshow(arr, cmap='gray')
    plt.title('Imagen procesada (28x28)')
    plt.axis('off')
    plt.show()

    # Predicción
    probs = modelo.predict(arr[np.newaxis, ...], verbose=0)[0]
    pred  = np.argmax(probs)

    print(f'🔮 El modelo cree que dibujaste el dígito: {pred}')
    print(f'   Confianza: {probs[pred]*100:.1f}%')

    plt.figure(figsize=(6, 3))
    plt.bar(range(10), probs * 100, color='steelblue', edgecolor='white')
    plt.bar(pred, probs[pred] * 100, color='green')
    plt.title('Probabilidades por dígito')
    plt.xlabel('Dígito')
    plt.ylabel('Probabilidad (%)')
    plt.xticks(range(10))
    plt.show()


print('💡 Dibujá un número en el canvas de arriba y presioná "Predecir"')
print('   Nota: asegurate de dibujar grande y centrado para mejores resultados')

---
## 📝 Resumen y Conclusiones

Completaste el ejercicio de reconocimiento de dígitos escritos a mano. 

### Lo que aprendiste:
| Concepto | Descripción |
|---|---|
| **Dataset MNIST** | 70.000 imágenes de dígitos en 28×28 px |
| **Normalización** | Escalar píxeles de [0-255] a [0-1] para mejor entrenamiento |
| **Red Neuronal Densa** | Capas totalmente conectadas (MLP) |
| **Dropout** | Técnica de regularización para evitar sobreajuste |
| **Softmax** | Convierte salidas en probabilidades |
| **Accuracy** | Porcentaje de aciertos del modelo |
| **Matriz de Confusión** | Visualización de aciertos y errores por clase |
| **CNN** | Arquitectura especializada para imágenes (desafío 2) |

### Resultados típicos esperados:
- **Red Densa (MLP)**: ~97-98% de accuracy
- **CNN**: ~99% de accuracy

### Para seguir aprendiendo:
- 📚 [TensorFlow Tutorials](https://www.tensorflow.org/tutorials)
- 📚 [Keras Documentation](https://keras.io/)
- 🎓 Explorá otros datasets: Fashion-MNIST, CIFAR-10